[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/piertotumlocomotor/robot_balance/blob/desafio1-rl/rl1_desafio1_cesaryepez.ipynb)

# Desafío 1 — Robot balanceador con aprendizaje por refuerzo
**Aprendizaje por Refuerzo I — MIA, FIUBA, 2026** · Repositorio: https://github.com/piertotumlocomotor/robot_balance/tree/desafio1-rl

**Problema.** Un robot de dos ruedas (péndulo invertido) tiene que mantenerse vertical moviendo sus
motores. Se construyó un simulador con los parámetros medidos del robot real y se entrenaron tres
agentes: **DQN**, **Double DQN** y **A2C**, comparados sobre el mismo entorno.

**Cómo correrlo.** *Entorno de ejecución → Ejecutar todas.* Con `RAPIDO = True` tarda ≈ 4 min;
con `False` entrena como en el informe (las semillas se reparten entre los núcleos disponibles).

In [1]:
%pip install -q gymnasium

## El modelo del robot

Péndulo invertido sobre dos ruedas, linealizado alrededor de la vertical:

$$J\,\ddot\theta = m g l\,\theta + \tau_{motor}(u) \quad\Longrightarrow\quad \ddot\theta = \omega_0^2\,\theta + K_U\,k(|u|)\,u$$

| Parámetro | Valor | Origen |
|---|---|---|
| $\omega_0$ | 8.0 rad/s | medido: video del robot colgado del eje (registro §7.22) |
| $J$ | 0.0159 kg·m² | medido: dos apoyos + $J = mgl/\omega_0^2$ (§7.23) |
| $m\,l$ | 0.1040 kg·m | medido: dos apoyos sobre balanza (§7.23) |
| $K_U$ | **0.043** rad/s² por unidad de duty | **supuesto** (ver abajo) |
| zona muerta M1 / M2 | 66 / 73 de duty | medida con el driver viejo, escalada igual que $K_U$ |
| tope de duty | 121 | 6 V del motor / 12.6 V de batería |

**Por qué $K_U$ es un supuesto.** El torque del motor con el driver nuevo (BTS7960) no se llegó a medir:
el ESP32 se dañó antes del ensayo. Se parte del torque medido con balanza usando el driver anterior
(L298N, dos motores a la vez): $4.89\times10^{-4}$ N·m por unidad de duty. El BTS7960 no tiene la caída de
~3 V del L298N, por lo que entrega entre 1.0× y 1.6× ese torque. Se adopta **1.4×** (medianamente optimista):

$$K_U = \frac{1.4 \times 4.89\times10^{-4}}{0.0159} \approx 0.043 \qquad\Rightarrow\qquad \theta_{umbral} = \frac{K_U \cdot 121}{\omega_0^2} \approx 4.7^\circ$$

$\theta_{umbral}$ es el máximo ángulo del que el robot puede recuperarse con todo el torque disponible.

## El problema como MDP

| Elemento | Definición |
|---|---|
| Estado $s$ | $(\theta,\ \dot\theta,\ u_{anterior})$, normalizados |
| Acción $a$ | 9 niveles de duty: −121, −105, −90, −75, 0, +75, +90, +105, +121 |
| Transición | la ecuación de arriba integrada a 1 ms; el agente decide cada 10 ms (100 Hz) |
| Recompensa | $r = 1 - (\theta/10^\circ)^2 - 0.05\,(u/121)^2 - 0.05\,(\Delta u/242)^2$ por paso |
| Fin del episodio | cae ($\lvert\theta\rvert > 10^\circ$, *terminated*) o pasan 5 s (*truncated*) |
| Estado inicial | $\lvert\theta_0\rvert \le 3.06^\circ$, $\lvert\dot\theta_0\rvert \le 0.15$ rad/s (error de colocación a mano medido) |
| Descuento | $\gamma = 0.99$ (horizonte ≈ 100 pasos = 1 s) |

El retorno máximo es ≈ 500 (500 pasos con recompensa ≈ 1). Un episodio en que el robot cae
rápido suma ≈ 30.

### Código del entorno (`entorno.py`)

In [2]:
%%writefile entorno.py
"""
Entorno de simulación del robot balanceador (planta nueva: 2× BTS7960 + chasis de balsa).

Modelo: péndulo invertido linealizado de 1 grado de libertad.

    J·θ̈ = m·g·l·θ + τ_motor(u)      →      θ̈ = ω₀²·θ + K_U·k(|u|)·u

    θ    inclinación del cuerpo [rad], 0 = vertical
    u    duty de PWM común a los dos motores (−121 … +121)
    k()  fracción del torque disponible según la zona muerta de cada motor

Lo que el modelo NO incluye: posición/velocidad de las ruedas, fuerza
contraelectromotriz, juego de la reductora, dinámica eléctrica del motor.
"""

import gymnasium as gym
import numpy as np
from gymnasium import spaces

# ---------------------------------------------------------------------------
# 1. Parámetros físicos
# ---------------------------------------------------------------------------
# Medidos en la planta nueva (registro de pruebas §7.22 y §7.23)
OMEGA0 = 8.0            # rad/s   frecuencia natural (video del robot colgado)
J = 0.0159              # kg·m²   momento de inercia respecto del eje
M_L = 0.1040            # kg·m    masa × distancia al centro de masa (dos apoyos)

# NO medido: el torque del motor con el driver nuevo (el ESP32 se quemó antes).
# Se parte del torque medido con balanza en la planta vieja, dos motores en
# brake a la vez (§7.20): τ = 0.0443 + 0.0339 N·m a duty 160 → 4.89e-4 N·m/duty.
# El BTS7960 (MOSFET) no tiene la caída de ~3 V del L298N, así que entrega más
# torque al mismo duty: entre 1.0× (pesimista) y 1.6× (optimista).
# SUPUESTO del proyecto: 1.4×, medianamente optimista.
FACTOR_DRIVER = 1.4
TORQUE_POR_DUTY = FACTOR_DRIVER * (0.0443 + 0.0339) / 160      # N·m por unidad de duty
K_U = TORQUE_POR_DUTY / J                                      # ≈ 0.043 rad/s² por unidad de duty

# Zona muerta: duty por debajo del cual el motor no hace fuerza. Medida en la
# planta vieja (92 y 102); como es un umbral de tensión, se escala con el driver.
ZONA_MUERTA_M1 = 92 / FACTOR_DRIVER     # ≈ 66
ZONA_MUERTA_M2 = 102 / FACTOR_DRIVER    # ≈ 73
PROP_M1 = 0.0443 / (0.0443 + 0.0339)    # M1 aporta el 57% del torque (M2 es más débil)
CAP = 121               # duty máximo: 6 V del motor / 12.6 V de batería llena (§8.5)

# Ángulo máximo recuperable con el torque disponible: θ_umbral = K_U·CAP/ω₀² ≈ 4.7°
THETA_UMBRAL = K_U * CAP / OMEGA0**2

# ---------------------------------------------------------------------------
# 2. Definición del MDP
# ---------------------------------------------------------------------------
DT = 0.01               # paso de control: 100 Hz
SUBPASOS = 10           # la física se integra en pasos de 1 ms
DURACION = 5.0          # s por episodio → 500 pasos
THETA_CAIDA = np.radians(10.0)      # más inclinado que esto = robot caído
THETA0_MAX = np.radians(3.06)       # error de colocación a mano medido (Test 10)
THETA_DOT0_MAX = 0.15               # rad/s, ídem

# Acciones: 9 niveles de duty. Todos los no nulos superan la zona muerta.
NIVELES_DUTY = np.array([-121, -105, -90, -75, 0, 75, 90, 105, 121], dtype=np.float64)


class RobotBalanceadorEnv(gym.Env):
    """
    Estado observado (normalizado):  [θ/θ_caída,  θ̇/(ω₀·θ_caída),  u_anterior/CAP]
    Acción:      índice 0..8 de NIVELES_DUTY
    Recompensa:  1 − (θ/θ_caída)² − 0.05·(u/CAP)² − 0.05·(Δu/2CAP)²   por paso; 0 si cae
    Fin:         terminated si |θ| > 10°;  truncated a los 5 s
    """

    def __init__(self, peso_u=0.05, peso_du=0.05):
        super().__init__()
        self.peso_u, self.peso_du = peso_u, peso_du
        self.niveles = NIVELES_DUTY / CAP                 # en fracción del cap
        self.max_pasos = int(round(DURACION / DT))
        self.action_space = spaces.Discrete(len(NIVELES_DUTY))
        self.observation_space = spaces.Box(-10.0, 10.0, shape=(3,), dtype=np.float32)

    # -- física ------------------------------------------------------------
    @staticmethod
    def torque_disponible(u_abs):
        """Fracción de K_U según cuántos motores superaron su zona muerta."""
        if u_abs < ZONA_MUERTA_M1:
            return 0.0
        if u_abs < ZONA_MUERTA_M2:
            return PROP_M1
        return 1.0

    def _integrar(self, u):
        aceleracion_motor = K_U * self.torque_disponible(abs(u)) * u
        dt = DT / SUBPASOS
        for _ in range(SUBPASOS):                     # Euler semi-implícito
            theta_ddot = OMEGA0**2 * self.theta + aceleracion_motor
            self.theta_dot += theta_ddot * dt
            self.theta += self.theta_dot * dt

    @staticmethod
    def es_recuperable(theta, theta_dot):
        """El modo inestable del péndulo es x = θ + θ̇/ω₀. Con el torque máximo
        solo se puede frenar si |x| < θ_umbral (cota optimista: ignora la zona muerta)."""
        return abs(theta + theta_dot / OMEGA0) < THETA_UMBRAL

    # -- interfaz Gymnasium -------------------------------------------------
    def _obs(self):
        return np.clip(np.array([self.theta / THETA_CAIDA,
                                 self.theta_dot / (OMEGA0 * THETA_CAIDA),
                                 self.u_anterior / CAP], dtype=np.float32), -10, 10)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        options = options or {}
        if "theta0" in options:                       # condición inicial fija (para graficar)
            self.theta = float(options["theta0"])
            self.theta_dot = float(options.get("theta_dot0", 0.0))
        else:                                         # sorteo: como colocarlo a mano
            while True:
                self.theta = self.np_random.uniform(-THETA0_MAX, THETA0_MAX)
                self.theta_dot = self.np_random.uniform(-THETA_DOT0_MAX, THETA_DOT0_MAX)
                if self.es_recuperable(self.theta, self.theta_dot):
                    break
        self.paso, self.u_anterior = 0, 0.0
        return self._obs(), {}

    def step(self, accion):
        u = NIVELES_DUTY[int(accion)]
        self._integrar(u)
        self.paso += 1

        recompensa = (1.0
                      - (self.theta / THETA_CAIDA) ** 2                       # estar vertical
                      - self.peso_u * (u / CAP) ** 2                          # gastar poca energía
                      - self.peso_du * ((u - self.u_anterior) / (2 * CAP)) ** 2)  # no "vibrar"
        self.u_anterior = u

        terminated = bool(abs(self.theta) > THETA_CAIDA)
        if terminated:
            recompensa = 0.0
        truncated = self.paso >= self.max_pasos
        info = {"theta": self.theta, "theta_dot": self.theta_dot, "u": u}
        return self._obs(), float(recompensa), terminated, truncated, info

Writing entorno.py


Verificación del entorno: API de Gymnasium, caída libre contra la solución analítica, y que el problema sea resoluble (PD a mano) pero no trivial (política al azar).

In [3]:
#@title test_entorno.py { display-mode: "form" }
open('test_entorno.py', 'w').write('"""\nVerificaciones del entorno, antes de entrenar nada encima.\nCorrer:  python test_entorno.py\n"""\n\nimport numpy as np\nfrom gymnasium.utils.env_checker import check_env\n\nfrom entorno import CAP, K_U, OMEGA0, THETA_UMBRAL, RobotBalanceadorEnv\n\n\ndef test_api_gymnasium():\n    check_env(RobotBalanceadorEnv(), skip_render_check=True)\n    print("OK  cumple la API de Gymnasium")\n\n\ndef test_parametros():\n    """K_U y θ_umbral coinciden con la cuenta a mano: sen θ = τ_max / (m·g·l)."""\n    a_mano = np.degrees(1.4 * (0.0443 + 0.0339) / 160 * CAP / (0.1040 * 9.81))\n    assert abs(np.degrees(THETA_UMBRAL) - a_mano) < 0.05\n    print(f"OK  K_U = {K_U:.4f} rad/s²/duty, θ_umbral = {np.degrees(THETA_UMBRAL):.2f}° (a mano {a_mano:.2f}°)")\n\n\ndef test_caida_libre():\n    """Sin motor, θ(t) = θ₀·cosh(ω₀·t). Valida el integrador y el signo."""\n    env = RobotBalanceadorEnv()\n    env.reset(seed=0, options={"theta0": np.radians(1.0)})\n    for _ in range(30):                                   # 0.3 s con duty 0 (acción 4)\n        _, _, _, _, info = env.step(4)\n    esperado = np.radians(1.0) * np.cosh(OMEGA0 * 0.3)\n    assert abs(info["theta"] - esperado) / esperado < 1e-2\n    print(f"OK  caída libre: {np.degrees(info[\'theta\']):.3f}° vs analítico {np.degrees(esperado):.3f}°")\n\n\ndef test_politicas_de_referencia():\n    """Resoluble (un PD a mano sobrevive) y no trivial (al azar cae enseguida)."""\n    env = RobotBalanceadorEnv()\n\n    def jugar(politica, n=50):\n        largos = []\n        for s in range(n):\n            obs, _ = env.reset(seed=s)\n            for paso in range(1, 501):\n                obs, _, term, trunc, _ = env.step(politica(obs))\n                if term or trunc:\n                    break\n            largos.append(paso if not term else paso - 500)   # negativo = cayó\n        return np.array(largos)\n\n    azar = jugar(lambda o: env.action_space.sample())\n    pd = jugar(lambda o: int(np.argmin(np.abs(env.niveles - np.clip(-(4.5 * o[0] + 3.0 * o[1]), -1, 1)))))\n    assert (pd == 500).all() and (azar < 0).all()\n    print(f"OK  PD a mano: 50/50 episodios completos | al azar: cae siempre, "\n          f"en {np.mean(azar + 500):.0f} pasos promedio")\n\n\nif __name__ == "__main__":\n    test_api_gymnasium()\n    test_parametros()\n    test_caida_libre()\n    test_politicas_de_referencia()\n')
print('escrito', 'test_entorno.py')

escrito test_entorno.py


In [4]:
%run test_entorno.py

OK  cumple la API de Gymnasium
OK  K_U = 0.0430 rad/s²/duty, θ_umbral = 4.66° (a mano 4.65°)
OK  caída libre: 5.579° vs analítico 5.557°
OK  PD a mano: 50/50 episodios completos | al azar: cae siempre, en 36 pasos promedio


### Agentes (`agentes.py`)
- **DQN**: red $Q(s,a)$, *replay buffer*, red objetivo, ε-greedy, pérdida de Huber.
- **Double DQN**: igual, pero la red principal **elige** la acción de $s'$ y la red objetivo la **evalúa** (función `objetivo_td`).
- **A2C**: actor (política) y crítico ($V(s)$), 8 entornos en paralelo, retornos de 16 pasos con GAE(λ=0.95).

In [5]:
%%writefile agentes.py
"""
Agentes para el robot balanceador — Aprendizaje por Refuerzo I, Desafío 1.

    DQN / Double DQN   basado en valor, off-policy (replay buffer + red objetivo)
    A2C                actor-crítico, on-policy (n-step returns + GAE, entornos en paralelo)

Todos usan exactamente el mismo MDP (entorno.py): la comparación es entre
algoritmos, no entre problemas.

Estructura de DQN igual a la del TP3 / clase 5; la de A2C igual a la práctica
de la clase 7 (RMSprop, init ortogonal, tanh, GAE opcional).
"""

import random
import time
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def media_movil(x, w=50):
    x = np.asarray(x, dtype=float)
    w = max(1, min(w, len(x)))
    return np.convolve(x, np.ones(w) / w, mode="valid")


# ===========================================================================
# DQN y Double DQN
# ===========================================================================

@dataclass
class HiperDQN:
    pasos: int = 150_000          # pasos de entorno (cada paso = 10 ms simulados)
    gamma: float = 0.99           # horizonte efectivo ~1/(1-γ) = 100 pasos = 1 s
    lr: float = 5e-4
    batch: int = 64
    buffer: int = 50_000
    inicio_aprendizaje: int = 2_000
    copiar_objetivo_cada: int = 1_000
    eps_inicial: float = 1.0
    eps_final: float = 0.02
    eps_pasos_decaimiento: int = 50_000
    oculta: int = 64
    double: bool = False


class ReplayBuffer:
    def __init__(self, capacidad, n_obs):
        self.capacidad, self.pos, self.tam = capacidad, 0, 0
        self.s = np.zeros((capacidad, n_obs), dtype=np.float32)
        self.s2 = np.zeros((capacidad, n_obs), dtype=np.float32)
        self.a = np.zeros(capacidad, dtype=np.int64)
        self.r = np.zeros(capacidad, dtype=np.float32)
        self.d = np.zeros(capacidad, dtype=np.float32)

    def guardar(self, s, a, r, s2, terminal):
        i = self.pos
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, terminal
        self.pos = (self.pos + 1) % self.capacidad
        self.tam = min(self.tam + 1, self.capacidad)

    def muestrear(self, n):
        idx = np.random.randint(0, self.tam, size=n)
        t = lambda x: torch.as_tensor(x[idx], device=device)
        return t(self.s), t(self.a).unsqueeze(1), t(self.r).unsqueeze(1), t(self.s2), t(self.d).unsqueeze(1)

    def __len__(self):
        return self.tam


class QNet(nn.Module):
    """MLP: estado (3) → Q(s, a) para cada acción."""
    def __init__(self, n_obs, n_acciones, oculta=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, oculta), nn.ReLU(),
            nn.Linear(oculta, oculta), nn.ReLU(),
            nn.Linear(oculta, n_acciones),
        )

    def forward(self, x):
        return self.net(x)


def objetivo_td(red, red_objetivo, r, s2, d, gamma, double):
    """y = r + γ·(1-terminal)·Q_objetivo(s', a*)
       DQN:        a* = argmax_a Q_objetivo(s', a)   (la misma red elige y evalúa)
       Double DQN: a* = argmax_a Q_red(s', a)        (una elige, la otra evalúa)"""
    with torch.no_grad():
        if double:
            a2 = red(s2).argmax(dim=1, keepdim=True)
            q_sig = red_objetivo(s2).gather(1, a2)
        else:
            q_sig = red_objetivo(s2).max(dim=1, keepdim=True)[0]
        return r + gamma * q_sig * (1.0 - d)


def entrenar_dqn(crear_env, hp: HiperDQN, seed=0, verbose=True):
    """Devuelve (red, pasos_fin_episodio, retornos_por_episodio)."""
    fijar_semilla(seed)
    env = crear_env()
    n_obs, n_acc = env.observation_space.shape[0], env.action_space.n
    red = QNet(n_obs, n_acc, hp.oculta).to(device)
    red_obj = QNet(n_obs, n_acc, hp.oculta).to(device)
    red_obj.load_state_dict(red.state_dict())
    opt = torch.optim.Adam(red.parameters(), lr=hp.lr)
    buffer = ReplayBuffer(hp.buffer, n_obs)

    s, _ = env.reset(seed=seed)
    ret_ep, pasos_fin, retornos = 0.0, [], []
    t0 = time.time()
    for paso in range(1, hp.pasos + 1):
        eps = max(hp.eps_final, hp.eps_inicial - (hp.eps_inicial - hp.eps_final) * paso / hp.eps_pasos_decaimiento)
        if random.random() < eps:
            a = random.randrange(n_acc)
        else:
            with torch.no_grad():
                a = int(red(torch.as_tensor(s, device=device).unsqueeze(0)).argmax(dim=1).item())

        s2, r, terminated, truncated, _ = env.step(a)
        # Solo 'terminated' (robot caído) corta el bootstrap: el truncamiento a
        # los 5 s es un límite del episodio, no un estado terminal del robot.
        buffer.guardar(s, a, r, s2, float(terminated))
        s, ret_ep = s2, ret_ep + r

        if len(buffer) >= max(hp.batch, hp.inicio_aprendizaje):
            bs, ba, br, bs2, bd = buffer.muestrear(hp.batch)
            q_sa = red(bs).gather(1, ba)
            y = objetivo_td(red, red_obj, br, bs2, bd, hp.gamma, hp.double)
            loss = F.smooth_l1_loss(q_sa, y)             # Huber
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(red.parameters(), 10.0)
            opt.step()

        if paso % hp.copiar_objetivo_cada == 0:
            red_obj.load_state_dict(red.state_dict())

        if terminated or truncated:
            pasos_fin.append(paso)
            retornos.append(ret_ep)
            if verbose and len(retornos) % 100 == 0:
                print(f"  paso {paso:7d} | ε={eps:.3f} | ep {len(retornos):4d} | "
                      f"media últ. 50: {np.mean(retornos[-50:]):6.1f}")
            ret_ep = 0.0
            s, _ = env.reset()
    if verbose:
        print(f"  {'Double DQN' if hp.double else 'DQN'} seed {seed}: {(time.time() - t0) / 60:.1f} min")
    return red, np.array(pasos_fin), np.array(retornos)


def politica_dqn(red):
    def pi(obs):
        with torch.no_grad():
            return int(red(torch.as_tensor(obs, device=device).unsqueeze(0)).argmax(dim=1).item())
    return pi


# ===========================================================================
# A2C (Advantage Actor-Critic)
# ===========================================================================

@dataclass
class HiperA2C:
    pasos: int = 300_000          # pasos de entorno totales (sumando los N entornos)
    n_envs: int = 8               # entornos en paralelo: decorrelacionan el batch (sin replay)
    n_steps: int = 16             # largo del rollout antes de cada actualización
    gamma: float = 0.99
    gae_lambda: float = 0.95      # 1.0 = n-step puro; <1 baja varianza a cambio de sesgo
    lr: float = 7e-4
    ent_coef: float = 0.01        # bonus de entropía: mantiene exploración
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    oculta: int = 64
    capas: int = 2


def _mlp(entrada, salida, oculta, capas, ganancia_salida):
    mods, d = [], entrada
    for _ in range(capas):
        lin = nn.Linear(d, oculta)
        nn.init.orthogonal_(lin.weight, np.sqrt(2)); nn.init.zeros_(lin.bias)
        mods += [lin, nn.Tanh()]
        d = oculta
    ult = nn.Linear(d, salida)
    nn.init.orthogonal_(ult.weight, ganancia_salida); nn.init.zeros_(ult.bias)
    return nn.Sequential(*mods, ult)


class ActorCritico(nn.Module):
    """Dos redes separadas: el actor da logits de π(a|s), el crítico estima V(s)."""
    def __init__(self, n_obs, n_acciones, oculta=64, capas=2):
        super().__init__()
        self.actor = _mlp(n_obs, n_acciones, oculta, capas, 0.01)   # política inicial ~uniforme
        self.critico = _mlp(n_obs, 1, oculta, capas, 1.0)

    def dist(self, x):
        return Categorical(logits=self.actor(x))

    def valor(self, x):
        return self.critico(x).squeeze(-1)


def ventajas_gae(r, v, v_ult, no_terminal, gamma, lam):
    """GAE(λ) hacia atrás sobre un rollout [T, N].
       δ_t = r_t + γ·V(s_{t+1})·(1-terminal) − V(s_t)
       A_t = δ_t + γλ·(1-fin_t)·A_{t+1}
    Con λ=1 es el retorno de n pasos menos V(s_t)."""
    T = r.shape[0]
    adv = torch.zeros_like(r)
    ultimo = torch.zeros_like(v_ult)
    for t in reversed(range(T)):
        v_sig = v_ult if t == T - 1 else v[t + 1]
        delta = r[t] + gamma * v_sig * no_terminal[t] - v[t]
        ultimo = delta + gamma * lam * no_terminal[t] * ultimo
        adv[t] = ultimo
    return adv, adv + v


def entrenar_a2c(crear_env, hp: HiperA2C, seed=0, verbose=True):
    """Devuelve (modelo, pasos_fin_episodio, retornos_por_episodio)."""
    fijar_semilla(seed)
    envs = [crear_env() for _ in range(hp.n_envs)]
    n_obs, n_acc = envs[0].observation_space.shape[0], envs[0].action_space.n
    modelo = ActorCritico(n_obs, n_acc, hp.oculta, hp.capas).to(device)
    opt = torch.optim.RMSprop(modelo.parameters(), lr=hp.lr, alpha=0.99, eps=1e-5)

    obs = np.stack([e.reset(seed=seed * 1000 + i)[0] for i, e in enumerate(envs)])
    ret_ep = np.zeros(hp.n_envs)
    pasos_fin, retornos = [], []
    n_updates = hp.pasos // (hp.n_envs * hp.n_steps)
    paso_global, t0 = 0, time.time()

    for upd in range(n_updates):
        for g in opt.param_groups:                       # lr con decaimiento lineal
            g["lr"] = hp.lr * (1 - upd / n_updates)

        b_obs, b_act, b_r, b_nt = [], [], [], []
        for _ in range(hp.n_steps):
            o_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
            with torch.no_grad():
                a = modelo.dist(o_t).sample()
            a_np = a.cpu().numpy()
            sig = np.empty_like(obs)
            r_t = np.zeros(hp.n_envs, dtype=np.float32)
            nt_t = np.ones(hp.n_envs, dtype=np.float32)
            for i, e in enumerate(envs):
                o2, r, term, trunc, _ = e.step(int(a_np[i]))
                ret_ep[i] += r
                if trunc and not term:
                    # Truncamiento por tiempo: el estado no es terminal, se
                    # bootstrapea con V(s_final) dentro de la recompensa.
                    with torch.no_grad():
                        r += hp.gamma * float(modelo.valor(torch.as_tensor(o2, dtype=torch.float32, device=device)))
                if term or trunc:
                    nt_t[i] = 0.0
                    pasos_fin.append(paso_global + i + 1)
                    retornos.append(ret_ep[i])
                    ret_ep[i] = 0.0
                    o2, _ = e.reset()
                r_t[i] = r
                sig[i] = o2
            b_obs.append(o_t); b_act.append(a)
            b_r.append(torch.as_tensor(r_t, device=device)); b_nt.append(torch.as_tensor(nt_t, device=device))
            obs = sig
            paso_global += hp.n_envs

        b_obs = torch.stack(b_obs)                       # [T, N, n_obs]
        b_act, b_r, b_nt = torch.stack(b_act), torch.stack(b_r), torch.stack(b_nt)
        dist = modelo.dist(b_obs)
        v = modelo.valor(b_obs)                           # [T, N]
        with torch.no_grad():
            v_ult = modelo.valor(torch.as_tensor(obs, dtype=torch.float32, device=device))
            adv, ret = ventajas_gae(b_r, v.detach(), v_ult, b_nt, hp.gamma, hp.gae_lambda)

        loss_actor = -(dist.log_prob(b_act) * adv).mean()     # gradiente de política con ventaja
        loss_critico = F.mse_loss(v, ret)
        entropia = dist.entropy().mean()
        loss = loss_actor + hp.vf_coef * loss_critico - hp.ent_coef * entropia
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), hp.max_grad_norm)
        opt.step()

        if verbose and (upd + 1) % max(1, n_updates // 10) == 0 and retornos:
            print(f"  paso {paso_global:7d} | ep {len(retornos):4d} | media últ. 50: "
                  f"{np.mean(retornos[-50:]):6.1f} | entropía {entropia.item():.2f}")
    if verbose:
        print(f"  A2C seed {seed}: {(time.time() - t0) / 60:.1f} min")
    return modelo, np.array(pasos_fin), np.array(retornos)


def politica_a2c(modelo):
    """Determinista para evaluar: la acción más probable."""
    def pi(obs):
        with torch.no_grad():
            return int(modelo.actor(torch.as_tensor(obs, dtype=torch.float32, device=device)).argmax().item())
    return pi


# ===========================================================================
# Referencia no-RL y evaluación
# ===========================================================================

def politica_pd(env, kp=4.5, kd=3.0):
    """PD a mano sobre la observación normalizada, cuantizado al nivel discreto
    más cercano. No aprende nada: sirve de vara para medir a los agentes."""
    def pi(obs):
        u = -(kp * obs[0] + kd * obs[1])
        return int(np.argmin(np.abs(env.niveles - np.clip(u, -1, 1))))
    return pi


def evaluar(politica, crear_env, n=200, semilla0=10_000):
    """Política fija (sin exploración) en episodios con semillas que el
    entrenamiento no usó."""
    env = crear_env()
    retornos, sobrevive, theta_abs, du_abs = [], [], [], []
    for k in range(n):
        obs, _ = env.reset(seed=semilla0 + k)
        total, u_prev, fin = 0.0, 0.0, False
        while not fin:
            obs, r, term, trunc, info = env.step(politica(obs))
            fin = term or trunc
            total += r
            theta_abs.append(abs(info["theta"]))
            du_abs.append(abs(info["u"] - u_prev)); u_prev = info["u"]
        retornos.append(total)
        sobrevive.append(not term)
    return {"retorno": float(np.mean(retornos)), "retorno_std": float(np.std(retornos)),
            "supervivencia": float(100 * np.mean(sobrevive)),
            "theta_medio_deg": float(np.degrees(np.mean(theta_abs))),
            "du_medio": float(np.mean(du_abs))}


def trayectoria(politica, crear_env, theta0_deg=3.0):
    """θ(t) y u(t) desde una inclinación inicial fija, para graficar."""
    env = crear_env()
    obs, _ = env.reset(seed=0, options={"theta0": np.radians(theta0_deg)})
    t, th, u, fin = [0.0], [theta0_deg], [0.0], False
    while not fin:
        obs, _, term, trunc, info = env.step(politica(obs))
        fin = term or trunc
        t.append(t[-1] + 0.01); th.append(np.degrees(info["theta"])); u.append(info["u"])
    return np.array(t), np.array(th), np.array(u)

Writing agentes.py


### Experimento: 3 agentes × N semillas, en paralelo

In [6]:
#@title experimentos.py { display-mode: "form" }
open('experimentos.py', 'w').write('"""\nExperimento del Desafío 1: DQN vs Double DQN vs A2C sobre el robot balanceador.\n\nCada agente se entrena con N semillas (en paralelo, un proceso por corrida) y se\nevalúa sin exploración.\n\nCorrer:   python experimentos.py                 (10 semillas, ≈ 25 min con 8 núcleos)\n          python experimentos.py --semillas 3    (≈ 10 min)\n          python experimentos.py --rapido        (≈ 3 min, para probar que todo anda)\nSalida:   figuras/*.png, resultados/resultados.json, resultados/modelos/*.pt\n"""\n\nimport argparse\nimport json\nimport os\nfrom concurrent.futures import ProcessPoolExecutor\nfrom dataclasses import asdict, replace\nfrom multiprocessing import get_context\n\nimport matplotlib\nmatplotlib.use("Agg")                         # solo se guardan PNG (también en Colab)\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\n\nfrom agentes import (ActorCritico, HiperA2C, HiperDQN, QNet, entrenar_a2c, entrenar_dqn, evaluar,\n                     media_movil, politica_a2c, politica_dqn, politica_pd, trayectoria)\nfrom entorno import NIVELES_DUTY, RobotBalanceadorEnv\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nFIG, RES = os.path.join(AQUI, "figuras"), os.path.join(AQUI, "resultados")\nCOLOR = {"DQN": "tab:blue", "Double DQN": "tab:red", "A2C": "tab:green", "PD a mano": "gray"}\nAGENTES = ("DQN", "Double DQN", "A2C")\nVENTANA = 50          # episodios de la media móvil\nUMBRAL = 450          # retorno que se considera "aprendido" para medir velocidad\n\n\ndef hiperparametros(rapido):\n    hp_dqn = HiperDQN(pasos=40_000, eps_pasos_decaimiento=15_000) if rapido else HiperDQN()\n    hp_a2c = HiperA2C(pasos=150_000) if rapido else HiperA2C()\n    return hp_dqn, hp_a2c\n\n\ndef crear_modelo(nombre):\n    n_obs, n_acc = 3, len(NIVELES_DUTY)\n    return ActorCritico(n_obs, n_acc) if nombre == "A2C" else QNet(n_obs, n_acc)\n\n\ndef a_politica(nombre, modelo):\n    return politica_a2c(modelo) if nombre == "A2C" else politica_dqn(modelo)\n\n\ndef correr(tarea):\n    """Una corrida completa (entrenar + evaluar). Se ejecuta en un proceso aparte."""\n    nombre, semilla, rapido, n_eval = tarea\n    torch.set_num_threads(1)                  # un núcleo por proceso: el paralelismo es entre corridas\n    hp_dqn, hp_a2c = hiperparametros(rapido)\n    if nombre == "A2C":\n        modelo, pasos, rets = entrenar_a2c(RobotBalanceadorEnv, hp_a2c, seed=semilla, verbose=False)\n    else:\n        hp = replace(hp_dqn, double=(nombre == "Double DQN"))\n        modelo, pasos, rets = entrenar_dqn(RobotBalanceadorEnv, hp, seed=semilla, verbose=False)\n    ev = evaluar(a_politica(nombre, modelo), RobotBalanceadorEnv, n=n_eval)\n    mm = media_movil(rets, VENTANA)\n    llega = np.nonzero(mm >= UMBRAL)[0]\n    ev["pasos_hasta_umbral"] = float(pasos[VENTANA - 1 + llega[0]]) if len(llega) else float("nan")\n    ev["episodios_entrenamiento"] = len(rets)\n    print(f"  {nombre:11s} semilla {semilla:2d}: retorno {ev[\'retorno\']:6.1f}, "\n          f"sobrevive {ev[\'supervivencia\']:5.1f}%", flush=True)\n    estado = {k: v.cpu() for k, v in modelo.state_dict().items()}\n    return nombre, semilla, pasos, rets, ev, estado\n\n\ndef graficar_convergencia(corridas, eje, archivo):\n    """Media entre semillas (línea) ± 1 desvío estándar (banda) del retorno,\n    suavizado con media móvil, contra episodios o contra pasos de entorno."""\n    fig, ax = plt.subplots(figsize=(8, 4.5))\n    for nombre, lista in corridas.items():\n        curvas = []\n        for pasos, rets in lista:\n            mm = media_movil(rets, VENTANA)\n            x = np.arange(VENTANA, len(rets) + 1) if eje == "episodios" else pasos[VENTANA - 1:]\n            curvas.append((x, mm))\n        grilla = np.linspace(max(x[0] for x, _ in curvas), min(x[-1] for x, _ in curvas), 300)\n        ys = np.array([np.interp(grilla, x, y) for x, y in curvas])\n        media, desvio = ys.mean(axis=0), ys.std(axis=0)\n        ax.fill_between(grilla, media - desvio, np.minimum(media + desvio, 500), color=COLOR[nombre], alpha=0.2)\n        ax.plot(grilla, media, color=COLOR[nombre], lw=2.2, label=f"{nombre} ({len(lista)} semillas)")\n    ax.axhline(500, color="gray", ls=":", lw=1)\n    ax.set_xlabel("Episodio" if eje == "episodios" else "Pasos de entorno (10 ms c/u)")\n    ax.set_ylabel(f"Retorno por episodio (media móvil {VENTANA})")\n    ax.set_ylim(0, 520); ax.grid(alpha=0.3); ax.legend(loc="lower right")\n    ax.set_title(f"Convergencia — retorno vs {eje} (media ± desvío entre semillas)")\n    plt.tight_layout(); plt.savefig(os.path.join(FIG, archivo), dpi=150); plt.close()\n\n\ndef graficar_semillas(evaluaciones, archivo):\n    """Un punto por semilla: cuánto varía el resultado final según el azar."""\n    fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))\n    for j, (clave, titulo) in enumerate((("retorno", "Retorno en evaluación"),\n                                         ("pasos_hasta_umbral", f"Pasos hasta retorno {UMBRAL}"))):\n        for i, nombre in enumerate(AGENTES):\n            y = [e[clave] for e in evaluaciones[nombre]]\n            x = i + np.random.default_rng(0).uniform(-0.12, 0.12, len(y))\n            axs[j].scatter(x, y, color=COLOR[nombre], alpha=0.7, s=28)\n            if np.isfinite(y).any():\n                axs[j].hlines(np.nanmean(y), i - 0.25, i + 0.25, color="black", lw=1.5)\n        axs[j].set_xticks(range(len(AGENTES)), AGENTES); axs[j].set_title(titulo); axs[j].grid(alpha=0.3, axis="y")\n    plt.tight_layout(); plt.savefig(os.path.join(FIG, archivo), dpi=150); plt.close()\n\n\ndef main(n_semillas=10, rapido=False, procesos=None):\n    os.makedirs(FIG, exist_ok=True)\n    os.makedirs(os.path.join(RES, "modelos"), exist_ok=True)\n    semillas = list(range(2 if rapido else n_semillas))\n    n_eval = 30 if rapido else 200\n    procesos = procesos or max(1, (os.cpu_count() or 2) - 1)\n\n    # 1) Entrenar y evaluar, todas las corridas en paralelo ---------------------\n    tareas = [(n, s, rapido, n_eval) for n in AGENTES for s in semillas]\n    print(f"{len(tareas)} corridas en {procesos} procesos...")\n    with ProcessPoolExecutor(procesos, mp_context=get_context("spawn")) as pool:\n        salidas = list(pool.map(correr, tareas))\n\n    corridas = {n: [] for n in AGENTES}\n    evaluaciones = {n: [] for n in AGENTES}\n    estados = {n: [] for n in AGENTES}\n    for nombre, semilla, pasos, rets, ev, estado in salidas:      # ya vienen en orden de semilla\n        corridas[nombre].append((pasos, rets))\n        evaluaciones[nombre].append(ev)\n        estados[nombre].append(estado)\n        torch.save(estado, os.path.join(RES, "modelos", f"{nombre.replace(\' \', \'_\')}_s{semilla}.pt"))\n\n    politicas = {"PD a mano": politica_pd(RobotBalanceadorEnv())}\n    for nombre in AGENTES:            # para graficar trayectorias: la mejor semilla de cada agente\n        mejor = int(np.argmax([e["retorno"] for e in evaluaciones[nombre]]))\n        modelo = crear_modelo(nombre); modelo.load_state_dict(estados[nombre][mejor])\n        politicas[nombre] = a_politica(nombre, modelo)\n    evaluaciones["PD a mano"] = [evaluar(politicas["PD a mano"], RobotBalanceadorEnv, n=n_eval)]\n\n    # 2) Tabla: media ± desvío entre semillas ----------------------------------\n    print(f"\\nEvaluación sin exploración ({n_eval} episodios nuevos por semilla), "\n          f"media ± desvío entre {len(semillas)} semillas:")\n    tabla = {}\n    for nombre, evs in evaluaciones.items():\n        fila = {}\n        for k in ("retorno", "supervivencia", "theta_medio_deg", "du_medio", "pasos_hasta_umbral"):\n            vals = [e[k] for e in evs if k in e and np.isfinite(e[k])]\n            fila[k] = (float(np.mean(vals)), float(np.std(vals))) if vals else None\n        tabla[nombre] = fila\n        f = lambda k, fmt: "—" if fila[k] is None else f"{fila[k][0]:{fmt}} ± {fila[k][1]:{fmt}}"\n        print(f"  {nombre:11s} retorno {f(\'retorno\', \'.1f\'):>14s} | sobrevive {f(\'supervivencia\', \'.1f\'):>12s}% | "\n              f"|θ| {f(\'theta_medio_deg\', \'.2f\'):>12s}° | |Δduty| {f(\'du_medio\', \'.1f\'):>12s} | "\n              f"pasos hasta {UMBRAL}: {f(\'pasos_hasta_umbral\', \'.0f\')}")\n\n    # 3) Figuras ----------------------------------------------------------------\n    graficar_convergencia(corridas, "episodios", "convergencia_episodios.png")\n    graficar_convergencia(corridas, "pasos", "convergencia_pasos.png")\n    graficar_semillas(evaluaciones, "dispersion_semillas.png")\n\n    fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)\n    for nombre, pi in politicas.items():\n        t, th, u = trayectoria(pi, RobotBalanceadorEnv, theta0_deg=3.0)\n        axs[0].plot(t, th, color=COLOR[nombre], label=nombre, lw=1.3)\n        axs[1].step(t, u, where="post", color=COLOR[nombre], lw=0.8)\n    axs[0].set_ylabel("θ [°]"); axs[1].set_ylabel("duty"); axs[1].set_xlabel("t [s]")\n    axs[0].legend(fontsize=8)\n    for ax in axs:\n        ax.grid(alpha=0.3)\n    fig.suptitle("Respuesta desde θ₀ = 3° (mejor semilla de cada agente)")\n    plt.tight_layout(); plt.savefig(os.path.join(FIG, "trayectorias.png"), dpi=150); plt.close()\n\n    nombre_json = "resultados_rapido.json" if rapido else "resultados.json"\n    with open(os.path.join(RES, nombre_json), "w") as fh:\n        json.dump({"semillas": semillas, "n_eval": n_eval, "umbral": UMBRAL,\n                   "hp_dqn": asdict(hiperparametros(rapido)[0]), "hp_a2c": asdict(hiperparametros(rapido)[1]),\n                   "evaluacion": tabla, "por_semilla": evaluaciones}, fh, indent=1, ensure_ascii=False)\n    print("\\nFiguras en", FIG)\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--semillas", type=int, default=10)\n    ap.add_argument("--rapido", action="store_true")\n    ap.add_argument("--procesos", type=int, default=None)\n    a = ap.parse_args()\n    main(a.semillas, a.rapido, a.procesos)\n    # En macOS el proceso auxiliar de multiprocessing a veces no termina y deja\n    # colgado el cierre del intérprete; todo ya está guardado, se sale directo.\n    os._exit(0)\n')
print('escrito', 'experimentos.py')

escrito experimentos.py


In [ ]:
RAPIDO = True      # True: 2 semillas cortas (≈ 4 min). False: entrenamiento completo
N_SEMILLAS = 3     # el informe usa 10 (≈ 30 min con 8 núcleos; en Colab, con 2 núcleos, varias horas)
import experimentos
experimentos.main(n_semillas=N_SEMILLAS, rapido=RAPIDO)

6 corridas en 1 procesos...


In [ ]:
from IPython.display import Image, display
for f in ["convergencia_episodios", "convergencia_pasos", "dispersion_semillas", "trayectorias"]:
    display(Image(f"figuras/{f}.png", width=720))

## Resultados y discusión

Experimento completo: 10 semillas por agente; evaluación sin exploración en 200 episodios nuevos por semilla (media ± desvío entre semillas).

| Agente | Retorno (máx. ≈ 500) | Sobrevive | \|θ\| medio | \|Δduty\| medio | Pasos hasta retorno 450 |
|---|---|---|---|---|---|
| DQN | 493.3 ± 3.9 | 100.0 ± 0.0 % | 0.59 ± 0.24° | 19.6 ± 20.9 | 72724 ± 6478 |
| Double DQN | 487.6 ± 7.7 | 100.0 ± 0.0 % | 0.80 ± 0.32° | 40.8 ± 27.5 | 80797 ± 6028 |
| A2C | 478.3 ± 5.6 | 99.9 ± 0.2 % | 0.31 ± 0.16° | 23.0 ± 5.0 | 166383 ± 28114 |
| PD a mano | 494.4 | 100.0 % | 0.66° | 32.4 | — |

**Discusión**

- **Los tres agentes aprenden a balancear el robot** en las 10 semillas. DQN y Double DQN sobreviven el 100 % de los episodios de evaluación; A2C, el 99.9 % (una caída en 2 000 episodios).
- **DQN es el más eficiente en datos**: llega a un retorno de 450 en ≈ 73 000 pasos, Double DQN en ≈ 81 000 y A2C en ≈ 166 000, con una dispersión entre semillas bastante mayor (Figura 5). DQN y Double DQN reutilizan cada transición muchas veces desde el *replay buffer*; A2C (*on-policy*) usa cada dato una sola vez.
- **Double DQN no mejora a DQN en este problema**: es algo más lento y obtiene un retorno un poco menor y más disperso (487.6 ± 7.7 contra 493.3 ± 3.9). Con 9 acciones y recompensas acotadas en [0, 1] la sobreestimación del máximo es chica, así que hay poco que corregir; una explicación posible de la leve demora es que su objetivo, más conservador, propaga el valor más despacio.
- **A2C deja al robot más derecho** (|θ| medio 0.31° contra 0.59° de DQN) pero lo hace con pulsos de motor a fondo (Figura 6), que la recompensa castiga; por eso su retorno es algo menor. Su curva de entrenamiento queda por debajo porque mientras entrena la política es aleatoria.
- **El PD ajustado a mano rinde igual que el mejor agente** (494.4 contra 493.3). El modelo es casi lineal y el problema es chico; el aporte del aprendizaje por refuerzo es llegar al mismo desempeño **sin conocer la ecuación**, solo con la recompensa.
- En todos los casos queda una **oscilación residual** de unas décimas de grado cerca de la vertical (Figura 6), por la falta de niveles de motor finos (sección 6).

**Inconvenientes y limitaciones**: ver el informe PDF entregado, secciones 6 y 7.

## Referencias

1. Sutton, R. S. y Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2.ª ed.). MIT Press. Cap. 6 (Q-learning), 9 (aproximación de funciones) y 13 (actor-crítico).
2. Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature* 518, 529–533. (DQN)
3. van Hasselt, H., Guez, A. y Silver, D. (2016). Deep Reinforcement Learning with Double Q-learning. *AAAI*. (Double DQN)
4. Mnih, V. et al. (2016). Asynchronous Methods for Deep Reinforcement Learning. *ICML*. (A3C/A2C)
5. Schulman, J. et al. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. *ICLR*. (GAE)
6. Towers, M. et al. (2024). Gymnasium: A Standard Interface for Reinforcement Learning Environments. arXiv:2407.17032.
7. Material de la materia (FIUBA, 2026): clase 5 (DQN), clase 7 (A2C), TP3.
8. Registro de pruebas del robot: `docs/conexiones-registro-pruebas.md` del repositorio (§7.20, §7.22, §7.23, §8.5).